# Canonical CRSP delisting reconstruction

This notebook is the canonical entry point for constructing the delisting-adjusted CRSP daily panel.

- `reference` reproduces exactly the 1990–2025 panel used in the thesis from the private 1,244-row reference ledger. It generates no random value.
- `seeded` generates every missing event-day `DelRet` with a user-selected base seed and a SHA-256-derived per-observation seed.
- `extend` preserves reference values for known historical events and applies the seeded rule only to genuinely new events.
- The private ledger is CRSP-derived and must not be committed to a public repository.

The default configuration validates the reference generation without writing a second 4 GB panel. Set `RUN_FULL_RECONSTRUCTION = True` only when a complete output Parquet is explicitly required. The canonical processed panel then feeds `notebooks/00_build_common_stock_source.ipynb`, which applies the point-in-time common-stock eligibility rules and prepares the reduced downstream source.

## Input contract

The input is not an untouched CRSP extract. It is a daily CRSP security panel previously merged on `PERMNO` with the CRSP delisting-event table (`DEL`). It must contain one security-date observation per row and the event fields `DelistingDt`, `DelRet`, `DelReasonType`, `DelActionType`, and `DelStatusType`. Observations strictly subsequent to the selected delisting date have already been removed. Missing event-day delisting returns have not yet been imputed, and `DlyRet` has not yet been compounded with the terminal return.

The expected local input is `data/crsp_daily_delistings_unprocessed.parquet`.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd / 'data' / 'RAW_DATA_PS', cwd, cwd.parent / 'RAW_DATA_PS']
RAW_DIR = next((p for p in candidates if (p / 'src' / 'delisting_panel_builder.py').exists()), None)
if RAW_DIR is None:
    raise FileNotFoundError('Cannot locate data/RAW_DATA_PS/src/delisting_panel_builder.py')

SRC_DIR = RAW_DIR / 'src'
sys.path.insert(0, str(SRC_DIR))
from delisting_panel_builder import reconstruct_panel, validate_ledger

DATA_DIR = RAW_DIR.parent
SOURCE = DATA_DIR / 'crsp_daily_delistings_unprocessed.parquet'
CANONICAL_PROCESSED = DATA_DIR / 'crsp_daily_shumway_delisting_processed.parquet'
PRIVATE_DIR = RAW_DIR / 'private'
LEDGER = PRIVATE_DIR / 'delret_reference_ledger_v1.parquet'
MANIFEST = PRIVATE_DIR / 'delret_reference_manifest_v1.json'

# Choose exactly one mode: 'reference', 'seeded', or 'extend'.
MODE = 'reference'
# Used only by 'seeded' and by new observations under 'extend'.
BASE_SEED = 42

if MODE == 'reference':
    output_name = 'crsp_daily_shumway_delisting_reference_rebuilt.parquet'
else:
    output_name = f'crsp_daily_shumway_delisting_{MODE}_seed{BASE_SEED}.parquet'
RECONSTRUCTED_OUTPUT = DATA_DIR / output_name

RUN_FULL_RECONSTRUCTION = False
VERIFY_FULL_FILE_HASHES = True

required_paths = [SOURCE]
if MODE in {'reference', 'extend'}:
    required_paths.append(LEDGER)
if MODE == 'reference':
    required_paths.append(CANONICAL_PROCESSED)
    if VERIFY_FULL_FILE_HASHES:
        required_paths.append(MANIFEST)

for required in required_paths:
    if not required.exists():
        raise FileNotFoundError(required)

print(f'RAW directory: {RAW_DIR}')
print(f'mode: {MODE}')
if MODE in {'seeded', 'extend'}:
    print(f'base seed: {BASE_SEED}')
else:
    print('base seed: not used in reference mode')

## 1. Validate the reference contract

Validation re-extracts the 1,244 missing event-day delisting returns from the aligned source and processed panels, checks the exact ledger contents, and optionally verifies the full-file SHA-256 digests. No panel is rewritten.

In [ ]:
if MODE == 'reference':
    validate_ledger(
        source_path=SOURCE,
        processed_path=CANONICAL_PROCESSED,
        ledger_path=LEDGER,
        manifest_path=MANIFEST if VERIFY_FULL_FILE_HASHES else None,
    )
elif MODE == 'extend':
    print('The reference ledger will be schema-validated during reconstruction.')
    print('New event keys will use the selected SHA-256 seeded rule.')
else:
    print('Seeded mode is independent of the private reference ledger.')
    print('All missing event-day DelRet values will use the selected seed.')

## 2. Optional full reconstruction

In `reference` mode, every historical missing `DelRet` must be supplied by the ledger. In `seeded` mode, all missing values are regenerated from the selected base seed. In `extend` mode, reference values are retained and only previously unseen events use the seeded rule. The canonical processed file is never overwritten automatically.

In [ ]:
if RUN_FULL_RECONSTRUCTION:
    reconstruct_panel(
        source_path=SOURCE,
        ledger_path=LEDGER if MODE in {'reference', 'extend'} else None,
        output_path=RECONSTRUCTED_OUTPUT,
        mode=MODE,
        base_seed=BASE_SEED,
    )
else:
    print('Validation only: full 4 GB reconstruction was not requested.')
    print('Set RUN_FULL_RECONSTRUCTION = True to create:')
    print(f'  {RECONSTRUCTED_OUTPUT}')